# Ethio-ASR 600M → CTranslate2 INT8 relay (fetch + convert on Kaggle's fast network)

One-shot relay: pull **`badrex/Ethio-ASR-multilingual-600M`** (2.4 GB fp32, normally
throttled to death on the dev Mac's network) using Kaggle's much faster upstream
bandwidth, convert it to the exact **CTranslate2 INT8** layout the product build
packages, and zip just that ~600 MB–1.1 GB directory back.

**Run All → wait for the last cell → click `Output → Download` → save
`ct2_600m_output.zip` on the Mac.** The Mac-side steps (unzip to
`tools/stage/model-ct2-int8-mu600`, honest A/B gate, decide swap) are in the
last cell's printed instructions.

The 600M model is `wav2vec2-bert` / hidden 1024 / 24 layers with **blank id 408**
(`[PAD]`) — identical to the shipped model's layout, so the repo converter
(`tools/make_model_ct2_int8.sh`) works **unchanged**; `model_meta.json` blank_id
408 matches the runtime loader.

In [ ]:
import os, subprocess, sys
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=False, text=True, **kw)
    if r.returncode:
        print(f"[fail] rc={r.returncode}: {' '.join(cmd)}")
        raise SystemExit(r.returncode)
    return r

HF_MODEL = "badrex/Ethio-ASR-multilingual-600M"
SRC = "/kaggle/working/mu600"            # downloaded fp32 checkpoint
DST = "/kaggle/working/model-ct2-int8-mu600"
ROOT = "/kaggle/working/amharic-caption"  # repo clone (public)
os.makedirs(SRC, exist_ok=True)

In [ ]:
# 1) clone the public repo so the converter is byte-identical to the build
if not os.path.isdir(ROOT):
    run(["git", "clone", "--depth", "1",
         "https://github.com/kaleb21-19/amharic_caption", ROOT])
print("repo ready")

# 2) top up packages (Kaggle image ships torch/numpy; we add the rest)
pip = [sys.executable, "-m", "pip", "install", "-q", "--no-input",
       "transformers>=4.52", "soundfile", "ctranslate2", "safetensors", "scipy"]
for _ in range(3):
    r = subprocess.run(pip, capture_output=True, text=True)
    if r.returncode == 0:
        break
    print(r.stderr[-400:])
else:
    raise SystemExit("pip install failed 3x")
import ctranslate2, transformers
print("ctranslate2 = ", ctranslate2.__version__, "| transformers =", transformers.__version__)

In [ ]:
# 3) FETCH the 2.4 GB fp32 checkpoint (fast on Kaggle's upstream)
from huggingface_hub import snapshot_download
snapshot_download(HF_MODEL, local_dir=SRC)
saf = os.path.getsize(os.path.join(SRC, "model.safetensors"))
print(f"[ok] model.safetensors = {saf/1e9:.2f} GB (expect 2.4245 GB)")
assert saf == 2424520168, "size mismatch — retry the fetch"

In [ ]:
# 4) CONVERT to CTranslate2 int8 using the exact repo converter
#    (writes mel_filters/window .npy, lm_head projection, vocab, model_meta
#     blank_id 408 — the runtime's expected layout).
env = dict(os.environ, MODEL_SRC=SRC, MODEL_DST=DST)
run(["bash", f"{ROOT}/tools/make_model_ct2_int8.sh"], env=env)

In [ ]:
# 5) SANITY — 600M vocab vs shipped: same [PAD]=408 blank, same alphabet size?
import json, hashlib
v600 = json.load(open(os.path.join(DST, "vocab.json")))
pad = [k for k, v in v600.items() if v == 408]
print("600M vocab len =", len(v600), "| id 408 =", pad)
print("dst files:", sorted(os.listdir(DST)))

In [ ]:
# 6) PACKAGE — zip ONLY the ct2 int8 dir for the return trip
import shutil, os
zpath = shutil.make_archive("/kaggle/working/ct2_600m_output", "zip", DST)
print("[ok] zip =", zpath, f"{os.path.getsize(zpath)/1e6:.0f} MB")
print("sha256 =", hashlib.sha256(open(zpath, 'rb').read()).hexdigest())

## On the Mac after the download

```bash
unzip ct2_600m_output.zip -d /tmp/mu600_zip
rm -rf tools/stage/model-ct2-int8-mu600
mv /tmp/mu600_zip/archive tools/stage/model-ct2-int8-mu600   # or the extracted dir name
# honest A/B gate, candidate vs shipped (§1.2g):
AMH_MODEL_DIR=tools/stage/model-ct2-int8-mu600 \
  tools/test/run_engine.sh --fixtures tools/test/fixtures_real --max-wer 0.15
AMH_MODEL_DIR=tools/stage/model-ct2-int8 \
  tools/test/run_engine.sh --fixtures tools/test/fixtures_real --max-wer 0.15
# keep the swap only if the candidate WER <= shipped on the same 20 clips.
```